## BERTuit

In [2]:
!pip install pysentimiento transformers datasets accelerate evaluate
!pip install ipdb
!pip install unidecode

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [3]:
import torch
from datasets import load_dataset
import unicodedata
import pandas as pd

import numpy as np
import evaluate
from pysentimiento.preprocessing import preprocess_tweet

from transformers import AutoModelForSequenceClassification, AutoTokenizer

from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix

import unicodedata

In [4]:
#keep slangs with only one word and less than 3 characters
def exclude_slangs(text):
    return len(text.split()) > 1 or len(text) <= 3

def apply_slang_exclusion(df_slangs):
    df_slangs = df_slangs[df_slangs.type.isin(["Jerga","expresión"])]
    df_slangs['slang'] = df_slangs['slang'].apply(lambda x: x.lower())
    df_slangs['exclude_slang'] = df_slangs['slang'].apply(lambda x: exclude_slangs(x))
    dfs = df_slangs[df_slangs.exclude_slang.isin([False])]
    return dfs

def remove_accents(text):
    return ''.join((c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn'))

Let's load a dataset -- in this case, a Spanish sentiment analysis dataset from CardiffNLP.

In [5]:
from datasets import load_dataset

ds = load_dataset("pyupeu/social-media-peruvian-sentiment",revision='4e797116f481d2a545e52cccbc29017fbf994fcc')
ds
ds_slangs = load_dataset("pyupeu/peruvian-slangs-dictionary")
ds_slangs

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/7574 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1894 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/2344 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'slang', 'type', 'short_description', 'description', 'source'],
        num_rows: 2344
    })
})

In [6]:
df_slangs = ds_slangs['train'].to_pandas()
dfs = apply_slang_exclusion(df_slangs)
slangs = dfs['slang'].values.tolist()
len(slangs)

1894

In [7]:
ds["train"].features

{'text': Value(dtype='string', id=None),
 'label': Value(dtype='int64', id=None),
 'label_name': Value(dtype='string', id=None),
 'text_original': Value(dtype='string', id=None),
 'tokenized_text': Value(dtype='string', id=None),
 'sent_token_length': Value(dtype='int64', id=None),
 'sent_bert_token_length': Value(dtype='int64', id=None),
 'char_count': Value(dtype='int64', id=None),
 'Character Count': Value(dtype='int64', id=None)}

In [8]:
ds["test"]["text"][:10]

['christian a. palomino christofher k. chqu carlos novoa chavez fiorella vilchez katherine ardiles podríamos hacer estooo ? 😭😭😭 salgamos parfavar...!',
 'la chuecona no pasa nada es gorda no tiene forma mil veces xoana un amor 😋',
 'para pulirla necesita una lija de fierro la más gruesa, o talves se canso de pulir 🤦🏻\u200d️🤦🏻\u200d️🤦🏻\u200d️🤣🤣🤣👍👍👍',
 '🤮🤮🤮 guacala ese maletero',
 'xabier cañola miñan que hago yo ahí? 😂😂😂',
 'recontra flaca. obvio métele al panetón pero también un par de papás al caldo  😂😂😂',
 'victor quiroz el cuto quiroz ,tu tbm tienes tu cachito pura vida 😂😂😂',
 'daniel daniela micaela vao! 😉',
 'wao eres un capo! como le haces para meter tanta comida jaja 😄😃😃😅 muy buen vídeo !! resalta lo mejor de la selva',
 'que rico, en argentina es un plato de cuando yo era chica y cuando yo alimentaba a mi familia. con el tiempo se ha perdido un poco pero para mi es un clásico. solo algunas variantes el pollo en 8 presas, morron colorado en lugar del ají amarillo que acá no tene

In [9]:
ds["test"]["label"][:10]

[1, 0, 2, 0, 1, 1, 1, 1, 2, 1]

## Load models

For this task, we use `robertuito-base-uncased` (there are other two versions: `robertuito-base-uncased`, and `robertuito-base-deacc`)

In [10]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "AIDA-UPM/BERTuit-base"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    ignore_mismatched_sizes=True,
    from_tf=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.model_max_length = 128

config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/529M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/483k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/283k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/821 [00:00<?, ?B/s]

## Preprocessing

Before tokenizing our model, we have to run the `preprocess_tweet` function to our data.


In [11]:
# preprocessed_ds = ds.map(lambda ex: {"text": preprocess_tweet(ex["text"], lang="es", demoji=False)})

preprocessed_ds = ds.map(lambda ex: {
    "text": remove_accents(preprocess_tweet(ex["text"], lang="es", demoji=False).lower())
})

Map:   0%|          | 0/7574 [00:00<?, ? examples/s]

Map:   0%|          | 0/2367 [00:00<?, ? examples/s]

Map:   0%|          | 0/1894 [00:00<?, ? examples/s]

In [12]:
preprocessed_ds["train"]["text"][:10]

['renzo crispin javier guere baluarte se pasaron de burros 🤣',
 'chino risas pa que llevas a tus maleteros 🤡🤡🤡',
 'estudia p chato 😵\u200d💫🙈',
 'no seas pendejo, jaja tatuaje borra entonces... 😂',
 'todas las autoridades y las actuales son culpables que kuelap se encuentre en esta situacion. son una tira de incapaces 😡',
 'lo que me preocupa es que el perrito esta tranquilo y feliz...espero no haya estado en una casa donde lo maltrataban... ojala termine en buenas manos!! natalia te vas al cielo sin escalas! 🙏🏻🙏🏻',
 '🔵🔴los productos de primera necesidades no pueden ser regulados gracias a la constitucion fujimorista del 93 articulo 62, 63  la constitucion prohibe la regulacion de precios, los productores pueden subirlos sin dar explicaciones a nadie. estas son las principales:  grupo romero -aportante de fuerza popular. duenos de alicorp,  produce fideos nicolini, don vittorio, alianza, lavaggi, aceite primor, cocinero, cil, friol, aceite tri a, capri, alacena; detergente sapolio, boli

## Tokenization

In [13]:
tokenized_ds = preprocessed_ds.map(
    lambda batch: tokenizer(batch["text"], padding=False, truncation=True),
    batched=True, batch_size=32
)

Map:   0%|          | 0/7574 [00:00<?, ? examples/s]

Map:   0%|          | 0/2367 [00:00<?, ? examples/s]

Map:   0%|          | 0/1894 [00:00<?, ? examples/s]

In [14]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'text_original', 'tokenized_text', 'sent_token_length', 'sent_bert_token_length', 'char_count', 'Character Count', 'input_ids', 'attention_mask'],
        num_rows: 7574
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name', 'text_original', 'tokenized_text', 'sent_token_length', 'sent_bert_token_length', 'char_count', 'Character Count', 'input_ids', 'attention_mask'],
        num_rows: 2367
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'text_original', 'tokenized_text', 'sent_token_length', 'sent_bert_token_length', 'char_count', 'Character Count', 'input_ids', 'attention_mask'],
        num_rows: 1894
    })
})

## Training

In [15]:
import numpy as np
import evaluate

f1_metric = evaluate.load("f1")
recall_metric = evaluate.load("recall")

def compute_metrics (eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis = -1)

    results = {}
    results.update(f1_metric.compute(predictions=preds, references = labels, average="macro"))
    results.update(recall_metric.compute(predictions=preds, references = labels, average="macro"))
    return results

In [16]:
tokenizer.add_tokens(slangs)

1887

In [17]:
model.resize_token_embeddings(len(tokenizer))#https://huggingface.co/docs/transformers/main_classes/tokenizer

Embedding(31845, 768)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    per_device_train_batch_size=32,
    warmup_ratio=0.1,
    learning_rate=5e-5,
    output_dir="test_trainer",
    do_eval=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    load_best_model_at_end=True,
    seed=1,
    group_by_length=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


{'eval_loss': 0.5717415809631348, 'eval_f1': 0.7215219613620437, 'eval_recall': 0.716198750787294, 'eval_runtime': 10.7898, 'eval_samples_per_second': 219.373, 'eval_steps_per_second': 27.433, 'epoch': 1.0}
{'eval_loss': 0.5799596905708313, 'eval_f1': 0.7253758318385476, 'eval_recall': 0.7212069260031129, 'eval_runtime': 11.4967, 'eval_samples_per_second': 205.885, 'eval_steps_per_second': 25.747, 'epoch': 2.0}
{'loss': 0.5676, 'grad_norm': 5.887819766998291, 'learning_rate': 3.212945590994372e-05, 'epoch': 2.109704641350211}
{'eval_loss': 0.7602986693382263, 'eval_f1': 0.7261348144840971, 'eval_recall': 0.7365617124752095, 'eval_runtime': 11.2613, 'eval_samples_per_second': 210.189, 'eval_steps_per_second': 26.285, 'epoch': 3.0}
{'eval_loss': 0.9171048402786255, 'eval_f1': 0.7307329078419512, 'eval_recall': 0.7304076707074453, 'eval_runtime': 11.2799, 'eval_samples_per_second': 209.841, 'eval_steps_per_second': 26.241, 'epoch': 4.0}
{'loss': 0.1722, 'grad_norm': 1.9054430723190308, 'l

TrainOutput(global_step=1185, training_loss=0.3212164697767813, metrics={'train_runtime': 319.4779, 'train_samples_per_second': 118.537, 'train_steps_per_second': 3.709, 'train_loss': 0.3212164697767813, 'epoch': 5.0})

In [ ]:
evaluation_results = trainer.evaluate(tokenized_ds["test"])

In [ ]:
predictions = trainer.predict(tokenized_ds["test"])
true_labels = tokenized_ds["test"]['label']

from sklearn.metrics import classification_report

predicted_labels = [np.argmax(pred) for pred in predictions[0]]
classification_rep = classification_report(true_labels, predicted_labels, digits=5)

print(classification_rep)